In [1]:
# Cell 1 - Import libraries
import spacy
import warnings
warnings.filterwarnings('ignore')

# Load english model
nlp = spacy.load('en_core_web_sm')

print("spaCy loaded successfully!")

spaCy loaded successfully!


In [4]:
# Cell 2 - Extract accident info from text
def extract_accident_info(text):
    doc = nlp(text)
    
    # Severity keywords
    high_keywords = ['unconscious', 'critical', 'not responding', 'bleeding heavily', 
                     'trapped', 'multiple', 'serious', 'fatal']
    medium_keywords = ['injured', 'hurt', 'wounded', 'broken', 'pain']
    low_keywords = ['minor', 'small', 'scratch', 'fender bender', 'slow']
    
    text_lower = text.lower()
    
    # Detect severity
    if any(word in text_lower for word in high_keywords):
        severity = 'High'
    elif any(word in text_lower for word in medium_keywords):
        severity = 'Medium'
    else:
        severity = 'Low'
    
    # Extract locations using spaCy
    locations = [ent.text for ent in doc.ents if ent.label_ in ['GPE', 'LOC', 'FAC']]
    
    # Extract numbers
    numbers = [ent.text for ent in doc.ents if ent.label_ == 'CARDINAL']
    
    return {
        'severity': severity,
        'locations': locations,
        'people_count': numbers,
        'original_text': text
    }

# Test it
test1 = "Bad accident on Anna Salai near Gemini flyover, 3 people injured, one is unconscious"
result = extract_accident_info(test1)
print("Input:", result['original_text'])
print("Severity:", result['severity'])
print("Locations found:", result['locations'])
print("Numbers found:", result['people_count'])

Input: Bad accident on Anna Salai near Gemini flyover, 3 people injured, one is unconscious
Severity: High
Locations found: ['Gemini']
Numbers found: ['3']


In [2]:
# Cell 3 - Voice input using Whisper
import whisper
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wav

# Load whisper model (tiny = fastest, good enough for us)
print("Loading Whisper model...")
whisper_model = whisper.load_model("tiny")
print("Whisper loaded!")

def record_audio(duration=5, sample_rate=16000):
    print(f"🎤 Recording for {duration} seconds... SPEAK NOW!")
    audio = sd.rec(int(duration * sample_rate), 
                   samplerate=sample_rate, 
                   channels=1, dtype='float32')
    sd.wait()
    print("✅ Recording done!")
    return audio, sample_rate

def transcribe_audio(audio, sample_rate):
    # Save temporarily
    wav.write('temp_audio.wav', sample_rate, audio)
    # Transcribe
    result = whisper_model.transcribe('temp_audio.wav')
    return result['text']

print("Voice system ready!")

Loading Whisper model...


100%|█████████████████████████████████████| 72.1M/72.1M [00:13<00:00, 5.52MiB/s]


Whisper loaded!
Voice system ready!


In [6]:
# Cell 4 - Test voice recording and transcription
print("Testing voice input...")
print("You will have 5 seconds to describe an accident")
print("Say something like: 'Accident on Anna Salai, 2 people injured'")
print()

# Record audio
audio, sr = record_audio(duration=5)

# Transcribe
text = transcribe_audio(audio, sr)
print("📝 You said:", text)

# Extract info
result = extract_accident_info(text)
print("\n🚨 Extracted Info:")
print("Severity:", result['severity'])
print("Locations:", result['locations'])
print("People count:", result['people_count'])

Testing voice input...
You will have 5 seconds to describe an accident
Say something like: 'Accident on Anna Salai, 2 people injured'

🎤 Recording for 5 seconds... SPEAK NOW!
✅ Recording done!
📝 You said:  uncertainty again another three people injured one is unconscious

🚨 Extracted Info:
Severity: High
Locations: []
People count: ['three', 'one']
